# Water / Flood Segmentation from 12-Channel Multispectral Satellite Imagery
### U-Net (from scratch, PyTorch) — Task 2, Week 1 Internship Deliverable



In [ ]:
%matplotlib inline
import os
import random

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score

# ----------------------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dataset Access



In [ ]:
!pip install -q gdown

In [ ]:
import gdown

FOLDER_URL = "https://drive.google.com/drive/folders/1GQss5oZhv-0dxoRtI5m_m_SPbm-E3vXJ"


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    print("Detected Google Colab -- downloading dataset with gdown ...")
    DATA_ROOT = "/content/water_seg_data"
    os.makedirs(DATA_ROOT, exist_ok=True)
    gdown.download_folder(url=FOLDER_URL, output=DATA_ROOT, quiet=False, use_cookies=False)
    DATA_DIR = DATA_ROOT

elif os.path.isdir("/kaggle/input"):
    print("Detected Kaggle -- using an attached dataset.")
    # Update this to the exact folder shown under /kaggle/input/<dataset-name>
    # once the dataset has been added via "Add Data".
    DATA_DIR = "/kaggle/input"

else:
    DATA_DIR = "./water_seg_data"
    print(f"Unrecognized environment -- expecting the dataset to already be available at {DATA_DIR}")

IMAGES_DIR = os.path.join(DATA_DIR, "data", "images")
LABELS_DIR = os.path.join(DATA_DIR, "data", "labels")

print("IMAGES_DIR:", IMAGES_DIR)
print("LABELS_DIR:", LABELS_DIR)

### Loading helpers



In [ ]:
BAND_NAMES = [
    "Coastal Aerosol",
    "Blue",
    "Green",
    "Red",
    "NIR",
    "SWIR1",
    "SWIR2",
    "QA Band",
    "MERIT DEM",
    "Copernicus DEM",
    "ESA WorldCover",
    "Water Occurrence Probability",
]
N_CHANNELS = len(BAND_NAMES)
R_IDX, G_IDX, B_IDX = 3, 2, 1  # Red, Green, Blue positions for RGB composites


def load_array(path):
    """Load a single array from disk, auto-detecting the file format."""
    ext = os.path.splitext(path)[1].lower()
    if ext == ".npy":
        return np.load(path)
    if ext == ".npz":
        with np.load(path) as npz:
            key = list(npz.keys())[0]
            return npz[key]
    if ext in (".tif", ".tiff"):
        try:
            import tifffile
            return tifffile.imread(path)
        except ImportError:
            try:
                import rasterio
                with rasterio.open(path) as src:
                    return src.read()  # (channels, H, W)
            except ImportError as e:
                raise ImportError(
                    "Reading .tif files needs 'tifffile' or 'rasterio'. "
                    "Install one, e.g. `!pip install tifffile`."
                ) from e
    raise ValueError(f"Unsupported file extension '{ext}' for file: {path}")


def ensure_channel_first(arr, n_channels=N_CHANNELS):
    """Return the array as (C, H, W) regardless of the on-disk axis order."""
    if arr.ndim != 3:
        raise ValueError(f"Expected a 3D array (H,W,C) or (C,H,W), got shape {arr.shape}")
    if arr.shape[0] == n_channels:
        return arr
    if arr.shape[-1] == n_channels:
        return np.transpose(arr, (2, 0, 1))
    raise ValueError(f"Could not find a channel axis of size {n_channels} in shape {arr.shape}")


def load_image(path):
    """Load one multispectral sample as float32, shape (12, 128, 128)."""
    arr = load_array(path).astype(np.float32)
    return ensure_channel_first(arr)


def load_mask(path):
    """Load one binary mask as float32 {0, 1}, shape (128, 128)."""
    arr = np.squeeze(load_array(path))
    if arr.ndim != 2:
        raise ValueError(f"Expected a 2D mask, got shape {arr.shape} for file: {path}")
    return (arr > 0).astype(np.float32)


def make_rgb_composite(img_chw, r_idx=R_IDX, g_idx=G_IDX, b_idx=B_IDX, percentile=(2, 98)):
    """Percentile-stretched RGB composite, for visualization only (not used by the model)."""
    rgb = np.stack([img_chw[r_idx], img_chw[g_idx], img_chw[b_idx]], axis=-1)
    out = np.zeros_like(rgb, dtype=np.float32)
    for i in range(3):
        band = rgb[..., i]
        lo, hi = np.percentile(band, percentile)
        out[..., i] = np.clip((band - lo) / (hi - lo + 1e-6), 0, 1)
    return out

## 2. Dataset Exploration

In [ ]:
image_files = sorted(os.listdir(IMAGES_DIR))
label_files = sorted(os.listdir(LABELS_DIR))

image_stems = {os.path.splitext(f)[0]: f for f in image_files}
label_stems = {os.path.splitext(f)[0]: f for f in label_files}

common_stems = sorted(set(image_stems) & set(label_stems))
missing_labels = sorted(set(image_stems) - set(label_stems))
missing_images = sorted(set(label_stems) - set(image_stems))

print(f"Image files found:  {len(image_files)}")
print(f"Label files found:  {len(label_files)}")
print(f"Matched pairs:      {len(common_stems)}")
if missing_labels:
    print(f"  Warning: {len(missing_labels)} image(s) have no matching label, e.g. {missing_labels[:3]}")
if missing_images:
    print(f"  Warning: {len(missing_images)} label(s) have no matching image, e.g. {missing_images[:3]}")

assert len(common_stems) > 0, "No matched image-mask pairs found -- check IMAGES_DIR/LABELS_DIR and file naming."

# Matching by filename stem (not by directory order) guarantees every image is
# paired with its own mask, which is what "correct image-mask pairing" requires.
image_paths = [os.path.join(IMAGES_DIR, image_stems[s]) for s in common_stems]
label_paths = [os.path.join(LABELS_DIR, label_stems[s]) for s in common_stems]

N_SAMPLES = len(common_stems)
print(f"\nTotal usable samples: {N_SAMPLES}")

In [ ]:
sample_img = load_image(image_paths[0])
sample_mask = load_mask(label_paths[0])

print("Single-sample image shape (after channel alignment):", sample_img.shape)
print("Single-sample image dtype:", sample_img.dtype)
assert sample_img.shape == (N_CHANNELS, 128, 128), "Unexpected image shape!"

print("\nSingle-sample mask shape:", sample_mask.shape)
print("Mask dtype:", sample_mask.dtype)
print("Unique mask values:", np.unique(sample_mask))
assert set(np.unique(sample_mask)).issubset({0.0, 1.0}), "Mask is not strictly binary!"

print("\nShape and label format verified: images are (12, 128, 128); masks are binary (128, 128).")

### 2.1 All 12 channels, one representative sample

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(17, 12))
for i, ax in enumerate(axes.flat):
    im = ax.imshow(sample_img[i], cmap="viridis")
    ax.set_title(f"{i+1}. {BAND_NAMES[i]}", fontsize=11)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.suptitle("All 12 Channels -- Single Representative Sample", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Representative images and their water masks

In [ ]:
n_show = 5
rng = np.random.default_rng(SEED)
show_idx = rng.choice(N_SAMPLES, size=min(n_show, N_SAMPLES), replace=False)

fig, axes = plt.subplots(len(show_idx), 2, figsize=(8, 4 * len(show_idx)))
if len(show_idx) == 1:
    axes = axes[None, :]

for row, idx in enumerate(show_idx):
    img = load_image(image_paths[idx])
    mask = load_mask(label_paths[idx])
    rgb = make_rgb_composite(img)

    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f"Sample {idx} -- RGB composite (bands 4-3-2)")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="Blues", vmin=0, vmax=1)
    axes[row, 1].set_title(f"Sample {idx} -- Water mask (1 = water)")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

### 2.3 Positive / negative pixel distribution

In [ ]:
total_pos = 0
total_neg = 0
for lp in label_paths:
    m = load_mask(lp)
    pos = int(m.sum())
    total_pos += pos
    total_neg += m.size - pos

total_pixels = total_pos + total_neg
print(f"Water pixels (1):      {total_pos:,}  ({100 * total_pos / total_pixels:.2f}%)")
print(f"Background pixels (0): {total_neg:,}  ({100 * total_neg / total_pixels:.2f}%)")

plt.figure(figsize=(5, 4))
bars = plt.bar(["Background (0)", "Water (1)"], [total_neg, total_pos],
                color=["#9e9e9e", "#1f77b4"])
plt.ylabel("Pixel count")
plt.title("Pixel-level class distribution across the full dataset")
for b, v in zip(bars, [total_neg, total_pos]):
    plt.text(b.get_x() + b.get_width() / 2, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 4. Preprocessing



In [ ]:
VAL_FRACTION = 0.2

train_idx, val_idx = train_test_split(
    np.arange(N_SAMPLES), test_size=VAL_FRACTION, random_state=SEED
)

print(f"Training samples:   {len(train_idx)}")
print(f"Validation samples: {len(val_idx)}")

In [ ]:
MAX_IMAGES_FOR_STATS = min(len(train_idx), 1000)  # cap for speed on very large datasets
PERCENTILE_RANGE = (1, 99)

stats_rng = np.random.default_rng(SEED)
stats_idx = stats_rng.choice(train_idx, size=MAX_IMAGES_FOR_STATS, replace=False)

band_pixel_values = [[] for _ in range(N_CHANNELS)]
for idx in stats_idx:
    img = load_image(image_paths[idx])
    for c in range(N_CHANNELS):
        band_pixel_values[c].append(img[c].ravel())

band_stats = np.zeros((N_CHANNELS, 2), dtype=np.float32)
for c in range(N_CHANNELS):
    all_vals = np.concatenate(band_pixel_values[c])
    band_stats[c] = np.percentile(all_vals, PERCENTILE_RANGE)
del band_pixel_values

print(f"Per-band statistics computed from {MAX_IMAGES_FOR_STATS} training samples "
      f"(p{PERCENTILE_RANGE[0]}/p{PERCENTILE_RANGE[1]}):\n")
for name, (lo, hi) in zip(BAND_NAMES, band_stats):
    print(f"  {name:28s}  low={lo:12.3f}   high={hi:12.3f}")

In [ ]:
def normalize_per_band(img, band_stats, eps=1e-6):
    """Per-band min-max normalization to [0, 1] using precomputed training statistics."""
    out = np.empty_like(img, dtype=np.float32)
    for c in range(img.shape[0]):
        lo, hi = band_stats[c]
        out[c] = np.clip((img[c] - lo) / (hi - lo + eps), 0.0, 1.0)
    return out


_check_img = load_image(image_paths[train_idx[0]])
_check_norm = normalize_per_band(_check_img, band_stats)
print("Normalized sample shape:", _check_norm.shape)
print(f"Normalized value range: [{_check_norm.min():.3f}, {_check_norm.max():.3f}]")
assert _check_norm.shape == (N_CHANNELS, 128, 128), "Preprocessing did not produce (12, 128, 128)!"
print("\nPreprocessing verified: output is (12, 128, 128), values in [0, 1].")
del _check_img, _check_norm

In [ ]:
class WaterSegDataset(Dataset):
    def __init__(self, indices, image_paths, label_paths, band_stats, augment=False):
        self.indices = indices
        self.image_paths = image_paths
        self.label_paths = label_paths
        self.band_stats = band_stats
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        img = load_image(self.image_paths[idx])
        mask = load_mask(self.label_paths[idx])
        img = normalize_per_band(img, self.band_stats)

        if self.augment:
            if random.random() < 0.5:  # horizontal flip
                img = img[:, :, ::-1].copy()
                mask = mask[:, ::-1].copy()
            if random.random() < 0.5:  # vertical flip
                img = img[:, ::-1, :].copy()
                mask = mask[::-1, :].copy()

        img_t = torch.from_numpy(img.astype(np.float32))
        mask_t = torch.from_numpy(mask.astype(np.float32)).unsqueeze(0)
        return img_t, mask_t


BATCH_SIZE = 16

train_dataset = WaterSegDataset(train_idx, image_paths, label_paths, band_stats, augment=True)
val_dataset = WaterSegDataset(val_idx, image_paths, label_paths, band_stats, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

imgs, masks = next(iter(train_loader))
print("Batch image shape:", tuple(imgs.shape))  # (B, 12, 128, 128)
print("Batch mask shape: ", tuple(masks.shape))  # (B, 1, 128, 128)

## 5. U-Net (from scratch)



In [ ]:
class DoubleConv(nn.Module):
    """(Conv3x3 -> BatchNorm -> ReLU) x 2"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net for 12-channel multispectral input and single-channel (binary) output.
    Encoder: 4 DoubleConv + maxpool stages. Bottleneck: 1 DoubleConv.
    Decoder: 4 transposed-conv upsample + skip-concat + DoubleConv stages.
    """

    def __init__(self, in_channels=N_CHANNELS, out_channels=1, base_channels=64):
        super().__init__()
        c = base_channels

        self.enc1 = DoubleConv(in_channels, c)
        self.enc2 = DoubleConv(c, c * 2)
        self.enc3 = DoubleConv(c * 2, c * 4)
        self.enc4 = DoubleConv(c * 4, c * 8)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.bottleneck = DoubleConv(c * 8, c * 16)

        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c * 16, c * 8)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c * 8, c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c * 4, c * 2)
        self.up1 = nn.ConvTranspose2d(c * 2, c, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c * 2, c)

        self.out_conv = nn.Conv2d(c, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out_conv(d1)  # raw logits, (B, 1, 128, 128)

In [ ]:
_sanity_model = UNet()
_x = torch.randn(2, N_CHANNELS, 128, 128)
_y = _sanity_model(_x)
print("Sanity check -- input:", tuple(_x.shape), "-> output:", tuple(_y.shape))
assert tuple(_y.shape) == (2, 1, 128, 128)
n_params = sum(p.numel() for p in _sanity_model.parameters())
print(f"Total trainable parameters: {n_params:,}")
del _sanity_model, _x, _y

### Loss function


In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        probs_flat = probs.view(probs.size(0), -1)
        targets_flat = targets.view(targets.size(0), -1)

        intersection = (probs_flat * targets_flat).sum(dim=1)
        dice_score = (2. * intersection + self.smooth) / (
            probs_flat.sum(dim=1) + targets_flat.sum(dim=1) + self.smooth
        )
        dice_loss = 1 - dice_score.mean()

        return bce_loss + dice_loss

## 6. Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, threshold=0.5, eps=1e-7):
    """Per-epoch validation loss + IoU, accumulated over the whole split (not averaged per-batch)."""
    model.eval()
    running_loss = 0.0
    tp = fp = fn = 0.0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        loss = criterion(logits, masks)
        running_loss += loss.item() * images.size(0)

        preds_bin = (torch.sigmoid(logits) > threshold).float()
        tp += (preds_bin * masks).sum().item()
        fp += (preds_bin * (1 - masks)).sum().item()
        fn += ((1 - preds_bin) * masks).sum().item()

    val_loss = running_loss / len(loader.dataset)
    iou = tp / (tp + fp + fn + eps)
    return val_loss, iou

In [ ]:
EPOCHS = 30
LEARNING_RATE = 1e-3
BEST_MODEL_PATH = "best_unet_water_segmentation.pt"

model = UNet(in_channels=N_CHANNELS, out_channels=1).to(device)
criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history = {"train_loss": [], "val_loss": [], "val_iou": []}
best_val_iou = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_iou = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_iou"].append(val_iou)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), BEST_MODEL_PATH)

    print(f"Epoch {epoch:3d}/{EPOCHS} | train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | val_IoU={val_iou:.4f}")

print(f"\nBest validation IoU during training: {best_val_iou:.4f}")

model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
print("Loaded best-checkpoint weights for evaluation below.")

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(epochs_range, history["train_loss"], label="Train loss")
axes[0].plot(epochs_range, history["val_loss"], label="Val loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss (BCE + Dice)")
axes[0].set_title("Training / Validation Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["val_iou"], color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("IoU")
axes[1].set_title("Validation IoU -- water class")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final-epoch validation IoU: {history['val_iou'][-1]:.4f}")
print(f"Best validation IoU (checkpoint used below): {best_val_iou:.4f}")

## 7. Evaluation



In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        probs = torch.sigmoid(model(images)).cpu().numpy()
        all_preds.append((probs > 0.5).astype(np.uint8).ravel())
        all_targets.append(masks.numpy().astype(np.uint8).ravel())

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

iou = jaccard_score(all_targets, all_preds, pos_label=1, average="binary", zero_division=0)
precision = precision_score(all_targets, all_preds, pos_label=1, average="binary", zero_division=0)
recall = recall_score(all_targets, all_preds, pos_label=1, average="binary", zero_division=0)
f1 = f1_score(all_targets, all_preds, pos_label=1, average="binary", zero_division=0)

print("Final validation-set evaluation (water class):")
print(f"  IoU:       {iou:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-score:  {f1:.4f}")

### Predictions vs. ground truth

In [ ]:
n_show = 6
pred_rng = np.random.default_rng(SEED + 1)
show_val_idx = pred_rng.choice(val_idx, size=min(n_show, len(val_idx)), replace=False)

fig, axes = plt.subplots(len(show_val_idx), 3, figsize=(13, 4 * len(show_val_idx)))
if len(show_val_idx) == 1:
    axes = axes[None, :]

model.eval()
with torch.no_grad():
    for row, idx in enumerate(show_val_idx):
        img = load_image(image_paths[idx])
        mask = load_mask(label_paths[idx])
        norm_img = normalize_per_band(img, band_stats)

        inp = torch.from_numpy(norm_img).unsqueeze(0).to(device)
        pred = (torch.sigmoid(model(inp)) > 0.5).float().cpu().numpy()[0, 0]

        rgb = make_rgb_composite(img)

        axes[row, 0].imshow(rgb)
        axes[row, 0].set_title(f"Sample {idx} -- image (RGB)")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask, cmap="Blues", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground truth mask")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred, cmap="Blues", vmin=0, vmax=1)
        axes[row, 2].set_title("Predicted mask")
        axes[row, 2].axis("off")

plt.tight_layout()
plt.show()